In [1]:
import os
import re
import pathlib as pl

In [2]:
os.chdir('..')
home = pl.Path(os.getcwd())

#user to set variables for the project. tagged as parameter for papermill runs
project = 'wy_fy22'


In [3]:
print('home is at: ',home)
home = pl.Path(home)
from src.hdf import *

inputs = home/'inputs'
outputs_base = home/'outputs'

export_folder = outputs_base/project/'trial_us_to_ds_events'
# target_export = export_folder/str('wy_gdg_'+target)

assert home.stem == '_code', 'restart kernel and rerun code'

home is at:  \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code


In [4]:
#location of upstream HUC dss files for connections between different HUC8s
transfer_dss_location = inputs/project/'transfer_dss'

In [5]:
#open dictionaries to understand which HUC the upstream one flows into and the downstream junction for each
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)
with open(inputs/project/'dictionaries'/'junc_res_sink_next_junc_down.json') as src:
    j_to_j = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_Junctions.json') as src:
    huc_js = json.load(src)
with open(inputs/project/'dictionaries'/'source_if_reservoir.json') as src:
    src_js = json.load(src)
with open(inputs/project/'dictionaries'/'hms_feature_origin_dict.json') as src:
    src_huc_hms = json.load(src)
with open(inputs/project/'dictionaries'/'outofscope_HUC10_to_HUC10.json') as src:
    out_of_scope_huc_inflows = json.load(src)
with open(inputs/project/'dictionaries'/'Junction_Subbasins.json') as src:
    j_connect_sub = json.load(src)
with open(inputs/project/'dictionaries'/'completed_event_dictionary.json') as cd:
    comp_dict = json.load(cd)

In [6]:
missing_events = {}
for ushuc, dshuc in huc_connect_huc.items():
    if dshuc in ['N/A','OUT']:
        pass 
    elif ushuc[:8] != dshuc[:8]:
        missing_events_list = []
        re_events = comp_dict[dshuc]
        for ri, events in re_events.items():
            for event in events:
                #print(dshuc,event)
                hit = 0
                #exist_dss = glob.glob(str(outputs_base/project/f'wy_gdg_{dshuc}*'/'[Hh]ydrology'/'*.dss'))
                exist_dss_transfer = glob.glob(str(inputs/project/'transfer_dss'/f'{ushuc[:8]}'/'*.dss'))
                for f in exist_dss_transfer:
                    if f.find(event.replace('-','_'))>=0:
                        #print(dshuc,ushuc,f)
                        hit+=1
                        #print(hit)
                if hit == 0:
                    #identify where it could be available
                    links = glob.glob(str(outputs_base/project/f'wy_gdg_{ushuc[:8]}*'/'[Hh]ydrology'/'*.dss'))
                    for link in links:
                        if link.find(event.replace('-','_'))>=0:
                            hit+=1
                            shutil.copy(link,str(inputs/project/'transfer_dss'/f'{ushuc[:8]}'/link.split('\\')[-1]))
                            print(dshuc,ushuc,link)
                            break
                if hit == 0:  
                    linksus = glob.glob(str(inputs/project/'us_dss'/f'HUC{dshuc[:8]}'/'*'/f'us_dss_HUC{ushuc[:8]}'/'*.dss'))
                    for linkus in linksus:
                        if linkus.find(event.replace('-','_').replace('R','R-'))>=0:
                            hit+=1
                            shutil.copy(linkus,str(inputs/project/'transfer_dss'/f'{ushuc[:8]}'/linkus.split('\\')[-1].replace('R-','R')))
                            print(dshuc,ushuc,linkus)
                            break
                            
                if hit == 0: 
                    missing_events_list.append(event)
                    #print(dshuc,ushuc[:8],event)
        missing_events[ushuc[:8]] = missing_events_list


1404010306 1404010406 \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010406\Hydrology\R5_Y043_E0002_output.dss
1404010604 1404010710 \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010709\Hydrology\R8_Y471_E0002_output.dss
1404010604 1404010710 \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\inputs\wy_fy22\us_dss\HUC14040106\1404010601\us_dss_HUC14040107\R-7_Y058_E0007_output.dss
1404010604 1404010710 \\us0236-ppfss01\shared_projects\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010707\Hydrology\R6_Y119_

In [7]:
import json
with open(export_folder/'needs2.json','w') as jj:
    json.dump(missing_events, jj)